In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import json

print("Libraries imported successfully.")

Libraries imported successfully.


### Define project paths

In [2]:
from pathlib import Path

# Project root
PROJECT_ROOT = Path(r"C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML")

# Input
RAW_FILE = PROJECT_ROOT / "data" / "phase1_fire_data" / "Phase_1_India_Hotspots_Combined.csv"

# Output
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed_data"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_FILE = PROCESSED_DIR / "firms_cleaned.csv"
REPORT_FILE = PROJECT_ROOT / "reports" / "preprocessing_report.json"

REPORT_FILE.parent.mkdir(parents=True, exist_ok=True)

print("Raw file:", RAW_FILE)
print("Output file:", CLEANED_FILE)

Raw file: C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML\data\phase1_fire_data\Phase_1_India_Hotspots_Combined.csv
Output file: C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML\data\processed_data\firms_cleaned.csv


### Check that the file exists


In [4]:
if RAW_FILE.exists():
    print("✅ FIRMS file found")
    print(f"File size: {RAW_FILE.stat().st_size / (1024**2):.2f} MB")
else:
    print("FIRMS file not found")

✅ FIRMS file found
File size: 64.00 MB


### Load FIRMS data

In [5]:
df = pd.read_csv(RAW_FILE)

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Rows: 702968
Columns: 12


,latitude,longitude,acq_date,acq_time,satellite,brightness,confidence,frp,daynight,version,detections_same_location_date,source_file
0,8.98492,77.60124,2025-01-01,801,NOAA-20 VIIRS,341.70,n,8.35,D,2,1,DL_FIRE_J1V-C2_812115.zip
1,9.00014,77.61005,2025-01-01,801,NOAA-20 VIIRS,330.68,n,2.68,D,2,1,DL_FIRE_J1V-C2_812115.zip
2,9.27809,78.04165,2025-01-01,801,NOAA-20 VIIRS,330.24,n,1.58,D,2,1,DL_FIRE_J1V-C2_812115.zip
3,9.56098,77.63283,2025-01-01,1949,NOAA-21 VIIRS,303.48,n,0.98,N,2.0NRT,1,DL_FIRE_J2V-C2_812116.zip
4,9.56323,77.63605,2025-01-01,1949,NOAA-21 VIIRS,303.55,n,1.23,N,2.0NRT,1,DL_FIRE_J2V-C2_812116.zip


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 702968 entries, 0 to 702967
Data columns (total 12 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   latitude                       702968 non-null  float64
 1   longitude                      702968 non-null  float64
 2   acq_date                       702968 non-null  str    
 3   acq_time                       702968 non-null  int64  
 4   satellite                      702968 non-null  str    
 5   brightness                     702968 non-null  float64
 6   confidence                     702968 non-null  str    
 7   frp                            702968 non-null  float64
 8   daynight                       702968 non-null  str    
 9   version                        702968 non-null  str    
 10  detections_same_location_date  702968 non-null  int64  
 11  source_file                    702968 non-null  str    
dtypes: float64(4), int64(2), str(6)
memory us

In [7]:
missing = df.isna().sum()

missing[missing > 0]

Series([], dtype: int64)

In [8]:
missing_percent = (df.isna().sum() / len(df) * 100).round(2)

missing_table = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": missing_percent
})

missing_table

,missing_count,missing_percent
latitude,0,0.0
longitude,0,0.0
acq_date,0,0.0
acq_time,0,0.0
satellite,0,0.0
brightness,0,0.0
confidence,0,0.0
frp,0,0.0
daynight,0,0.0
version,0,0.0


In [9]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


### Convert important columns

In [10]:
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
df["frp"] = pd.to_numeric(df["frp"], errors="coerce")
df["brightness"] = pd.to_numeric(df["brightness"], errors="coerce")

df["acq_date"] = pd.to_datetime(
    df["acq_date"],
    errors="coerce"
)

print(df.dtypes)

latitude                                float64
longitude                               float64
acq_date                         datetime64[us]
acq_time                                  int64
satellite                                   str
brightness                              float64
confidence                                  str
frp                                     float64
daynight                                    str
version                                     str
detections_same_location_date             int64
source_file                                 str
dtype: object


In [11]:
invalid_lat = ~df["latitude"].between(-90, 90)
invalid_lon = ~df["longitude"].between(-180, 180)

print("Invalid latitude:", invalid_lat.sum())
print("Invalid longitude:", invalid_lon.sum())

Invalid latitude: 0
Invalid longitude: 0


### Check invalid FRP

In [12]:
invalid_frp = df["frp"] < 0

print("Negative FRP values:", invalid_frp.sum())

Negative FRP values: 0


### Create cleaning mask

In [13]:
valid_mask = (
    df["latitude"].notna()
    & df["longitude"].notna()
    & df["acq_date"].notna()
    & df["frp"].notna()
    & df["latitude"].between(-90, 90)
    & df["longitude"].between(-180, 180)
    & (df["frp"] >= 0)
)

print("Valid rows:", valid_mask.sum())
print("Rows to remove:", (~valid_mask).sum())

Valid rows: 702968
Rows to remove: 0


### Create cleaned dataset

In [14]:
df_clean = df.loc[valid_mask].copy()

print("Original rows:", len(df))
print("Cleaned rows:", len(df_clean))
print("Removed rows:", len(df) - len(df_clean))

Original rows: 702968
Cleaned rows: 702968
Removed rows: 0


### Remove exact duplicates

In [15]:
before_duplicates = len(df_clean)

df_clean = df_clean.drop_duplicates().copy()

after_duplicates = len(df_clean)

print("Duplicates removed:", before_duplicates - after_duplicates)
print("Final rows:", after_duplicates)

Duplicates removed: 0
Final rows: 702968


### Sort the data

In [16]:
df_clean = df_clean.sort_values(
    by=["acq_date", "latitude", "longitude"]
).reset_index(drop=True)

df_clean.head()

,latitude,longitude,acq_date,acq_time,satellite,brightness,confidence,frp,daynight,version,detections_same_location_date,source_file
0,8.98492,77.60124,2025-01-01,801,NOAA-20 VIIRS,341.70,n,8.35,D,2,1,DL_FIRE_J1V-C2_812115.zip
1,9.00014,77.61005,2025-01-01,801,NOAA-20 VIIRS,330.68,n,2.68,D,2,1,DL_FIRE_J1V-C2_812115.zip
2,9.27809,78.04165,2025-01-01,801,NOAA-20 VIIRS,330.24,n,1.58,D,2,1,DL_FIRE_J1V-C2_812115.zip
3,9.56098,77.63283,2025-01-01,1949,NOAA-21 VIIRS,303.48,n,0.98,N,2.0NRT,1,DL_FIRE_J2V-C2_812116.zip
4,9.56323,77.63605,2025-01-01,1949,NOAA-21 VIIRS,303.55,n,1.23,N,2.0NRT,1,DL_FIRE_J2V-C2_812116.zip


### Save cleaned FIRMS data

In [17]:
df_clean.to_csv(
    CLEANED_FILE,
    index=False
)

print("✅ Cleaned FIRMS data saved:")
print(CLEANED_FILE)

✅ Cleaned FIRMS data saved:
C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML\data\processed_data\firms_cleaned.csv


Generate preprocessing report

In [18]:
report = {
    "input_file": str(RAW_FILE),
    "output_file": str(CLEANED_FILE),

    "raw_row_count": int(len(df)),

    "rows_removed_invalid": int((~valid_mask).sum()),

    "rows_removed_duplicates": int(
        before_duplicates - after_duplicates
    ),

    "final_row_count": int(len(df_clean)),

    "date_min": str(df_clean["acq_date"].min().date()),
    "date_max": str(df_clean["acq_date"].max().date()),

    "columns": list(df_clean.columns),

    "cleaning_rules": [
        "Remove records with missing latitude",
        "Remove records with missing longitude",
        "Remove records with invalid latitude",
        "Remove records with invalid longitude",
        "Remove records with invalid acquisition date",
        "Remove records with missing FRP",
        "Remove records with negative FRP",
        "Remove exact duplicate rows"
    ]
}

with open(REPORT_FILE, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=4)

print("✅ Preprocessing report saved:")
print(REPORT_FILE)

✅ Preprocessing report saved:
C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML\reports\preprocessing_report.json


In [19]:
print("Raw rows:", report["raw_row_count"])
print("Invalid rows removed:", report["rows_removed_invalid"])
print("Duplicate rows removed:", report["rows_removed_duplicates"])
print("Final cleaned rows:", report["final_row_count"])
print("Date range:", report["date_min"], "→", report["date_max"])

Raw rows: 702968
Invalid rows removed: 0
Duplicate rows removed: 0
Final cleaned rows: 702968
Date range: 2025-01-01 → 2025-03-19
